# Specify the spatial variables with `lat_key` and `lon_key`

This notebook demonstrates how to tell the spatial transforms which coordinates hold the latitude and longitude values using the `lat_key` and `lon_key` arguments.

By default the spatial methods detect the latitude and longitude coordinates automatically from the metadata of the data object (via the CF `standard_name`/`axis` attributes or a set of recognised names such as `latitude`/`longitude` and `lat`/`lon`). When your data uses non-standard coordinate names, or the automatic detection picks the wrong coordinate, you can pass `lat_key` and `lon_key` explicitly.

In [ ]:
from earthkit import data as ekd
from earthkit import transforms as ekt
from earthkit.transforms._tools import earthkit_remote_test_data_file

remote_era5_file = earthkit_remote_test_data_file("era5-Europe-sfc-2m-temperature-3deg-2015-2017.grib")
era5_data = ekd.from_source("url", remote_era5_file)
ds = era5_data.to_xarray()

ds

## Create data with non-standard coordinate names

To simulate a dataset that the automatic detection cannot resolve, we rename the latitude and longitude coordinates to `y_coord` and `x_coord` and drop the CF `standard_name` and `axis` attributes.

In [ ]:
spatial_info = ekt._tools.get_spatial_info(ds)
lat_key, lon_key = spatial_info["lat_key"], spatial_info["lon_key"]

ds_renamed = ds.rename({lat_key: "y_coord", lon_key: "x_coord"})
for coord in ("y_coord", "x_coord"):
    ds_renamed[coord].attrs.pop("standard_name", None)
    ds_renamed[coord].attrs.pop("axis", None)
    ds_renamed[coord].attrs.pop("long_name", None)
    ds_renamed[coord].attrs.pop("units", None)

ds_renamed

## Provide the bespoke `lat_key` and `lon_key`

Because the spatial coordinates are now named `y_coord` and `x_coord`, we pass `lat_key="y_coord"` and `lon_key="x_coord"` to `spatial.reduce` so that it aggregates over the correct spatial dimensions.

In [ ]:
area = {"north": 55, "south": 45, "east": 20, "west": 5}

area_mean = ekt.spatial.reduce(
    ds_renamed,
    area=area,
    how="mean",
    weights="latitude",
    lat_key="y_coord",
    lon_key="x_coord",
)

area_mean